# Day 09. Exercise 00
# Regularization

## 0. Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, mean_squared_error, silhouette_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import make_pipeline
import joblib
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/dayofweek.csv')
X = df.drop(columns='dayofweek')
y = df['dayofweek']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [3]:
%%time

logreg = LogisticRegression(random_state=21, fit_intercept=False)
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    logreg.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, logreg.predict(X_tr))
    score_val = accuracy_score(y_val, logreg.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 0.62887 | valid - 0.73964
train - 0.65260 | valid - 0.61538
train - 0.65458 | valid - 0.60947
train - 0.63612 | valid - 0.54438
train - 0.64469 | valid - 0.63314
train - 0.64601 | valid - 0.57988
train - 0.62846 | valid - 0.57143
train - 0.64493 | valid - 0.61905
train - 0.63636 | valid - 0.60119
train - 0.64032 | valid - 0.61310
Average accuracy on crossval is 0.61267
Std is 0.04916
CPU times: total: 1 s
Wall time: 698 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [4]:
%%time

logreg = LogisticRegression(random_state=21, fit_intercept=False, penalty='none', max_iter=1000)
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_val = scaler.transform(X_val)

    logreg.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, logreg.predict(X_tr))
    score_val = accuracy_score(y_val, logreg.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 0.65854 | valid - 0.75148
train - 0.66249 | valid - 0.62722
train - 0.65722 | valid - 0.62130
train - 0.66183 | valid - 0.57396
train - 0.66711 | valid - 0.66272
train - 0.66974 | valid - 0.61538
train - 0.65744 | valid - 0.61905
train - 0.65547 | valid - 0.63095
train - 0.65876 | valid - 0.60714
train - 0.67260 | valid - 0.64286
Average accuracy on crossval is 0.63521
Std is 0.0445
CPU times: total: 2.75 s
Wall time: 1.75 s


In [5]:
%%time

logreg = LogisticRegression(
    penalty='l1', 
    C=1, 
    solver='saga',
    random_state=21, 
    max_iter=1000
)
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    logreg.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, logreg.predict(X_tr))
    score_val = accuracy_score(y_val, logreg.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 0.62821 | valid - 0.73373
train - 0.65260 | valid - 0.61538
train - 0.65194 | valid - 0.61538
train - 0.63546 | valid - 0.53846
train - 0.64601 | valid - 0.62722
train - 0.65063 | valid - 0.57988
train - 0.63307 | valid - 0.57143
train - 0.64559 | valid - 0.62500
train - 0.63636 | valid - 0.61310
train - 0.64229 | valid - 0.61310
Average accuracy on crossval is 0.61327
Std is 0.04828
CPU times: total: 7.7 s
Wall time: 8 s


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [6]:
svc = SVC(probability=True, random_state=21, kernel='linear')
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    svc.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, svc.predict(X_tr))
    score_val = accuracy_score(y_val, svc.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 0.69347 | valid - 0.76923
train - 0.71127 | valid - 0.68639
train - 0.70138 | valid - 0.67456
train - 0.69941 | valid - 0.60947
train - 0.70073 | valid - 0.69822
train - 0.70666 | valid - 0.72781
train - 0.70026 | valid - 0.65476
train - 0.71146 | valid - 0.63690
train - 0.69565 | valid - 0.68452
train - 0.71080 | valid - 0.64286
Average accuracy on crossval is 0.67847
Std is 0.04415


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [7]:
svc = SVC(probability=True, random_state=21, C=1, kernel='linear')
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    svc.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, svc.predict(X_tr))
    score_val = accuracy_score(y_val, svc.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 0.69347 | valid - 0.76923
train - 0.71127 | valid - 0.68639
train - 0.70138 | valid - 0.67456
train - 0.69941 | valid - 0.60947
train - 0.70073 | valid - 0.69822
train - 0.70666 | valid - 0.72781
train - 0.70026 | valid - 0.65476
train - 0.71146 | valid - 0.63690
train - 0.69565 | valid - 0.68452
train - 0.71080 | valid - 0.64286
Average accuracy on crossval is 0.67847
Std is 0.04415


In [8]:
svc = SVC(probability=True, random_state=21, C=0.5, kernel='linear')
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    svc.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, svc.predict(X_tr))
    score_val = accuracy_score(y_val, svc.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 0.67304 | valid - 0.75148
train - 0.68095 | valid - 0.67456
train - 0.68029 | valid - 0.66272
train - 0.67633 | valid - 0.55030
train - 0.68952 | valid - 0.68639
train - 0.67831 | valid - 0.68639
train - 0.67787 | valid - 0.62500
train - 0.68775 | valid - 0.61905
train - 0.67787 | valid - 0.67857
train - 0.68906 | valid - 0.64286
Average accuracy on crossval is 0.65773
Std is 0.0505


In [9]:
svc = SVC(probability=True, random_state=21, C=20, kernel='linear')
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    svc.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, svc.predict(X_tr))
    score_val = accuracy_score(y_val, svc.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 0.77060 | valid - 0.79882
train - 0.77983 | valid - 0.72781
train - 0.77587 | valid - 0.76923
train - 0.78576 | valid - 0.71006
train - 0.77521 | valid - 0.81657
train - 0.77521 | valid - 0.76923
train - 0.77404 | valid - 0.71429
train - 0.77339 | valid - 0.70833
train - 0.78986 | valid - 0.75595
train - 0.77866 | valid - 0.70833
Average accuracy on crossval is 0.74786
Std is 0.03792


## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [10]:
model = DecisionTreeClassifier(max_depth=10, random_state=21)
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, model.predict(X_tr))
    score_val = accuracy_score(y_val, model.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 0.80883 | valid - 0.78107
train - 0.82202 | valid - 0.76331
train - 0.81740 | valid - 0.73964
train - 0.82334 | valid - 0.75740
train - 0.81279 | valid - 0.78698
train - 0.82136 | valid - 0.82249
train - 0.81423 | valid - 0.72619
train - 0.81950 | valid - 0.72024
train - 0.82411 | valid - 0.76190
train - 0.82543 | valid - 0.75595
Average accuracy on crossval is 0.76152
Std is 0.02869


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [11]:
model = DecisionTreeClassifier(max_depth=15, random_state=21, min_samples_split=3, ccp_alpha=0)
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, model.predict(X_tr))
    score_val = accuracy_score(y_val, model.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 0.93804 | valid - 0.86982
train - 0.94265 | valid - 0.87574
train - 0.95056 | valid - 0.86391
train - 0.95452 | valid - 0.85207
train - 0.94990 | valid - 0.86982
train - 0.95254 | valid - 0.92308
train - 0.94137 | valid - 0.83333
train - 0.94993 | valid - 0.84524
train - 0.94401 | valid - 0.83929
train - 0.94401 | valid - 0.83929
Average accuracy on crossval is 0.86116
Std is 0.02504


In [12]:
model = DecisionTreeClassifier(max_depth=20, random_state=21, min_samples_split=3, min_samples_leaf=1, max_leaf_nodes=100000, ccp_alpha=0)
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, model.predict(X_tr))
    score_val = accuracy_score(y_val, model.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 0.98616 | valid - 0.90533
train - 0.98220 | valid - 0.89941
train - 0.98418 | valid - 0.89349
train - 0.97693 | valid - 0.88757
train - 0.98550 | valid - 0.89941
train - 0.98088 | valid - 0.92308
train - 0.98353 | valid - 0.88095
train - 0.98485 | valid - 0.87500
train - 0.98485 | valid - 0.85119
train - 0.98024 | valid - 0.85119
Average accuracy on crossval is 0.88666
Std is 0.02175


In [13]:
model = DecisionTreeClassifier(max_depth=25, random_state=21, min_samples_split=3, min_samples_leaf=1, max_leaf_nodes=100000, ccp_alpha=0)
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, model.predict(X_tr))
    score_val = accuracy_score(y_val, model.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 0.99077 | valid - 0.90533


train - 0.99143 | valid - 0.89941
train - 0.99143 | valid - 0.89941
train - 0.99275 | valid - 0.91716
train - 0.99143 | valid - 0.89349
train - 0.99077 | valid - 0.94083
train - 0.99012 | valid - 0.88095
train - 0.99078 | valid - 0.88095
train - 0.99209 | valid - 0.86310
train - 0.99144 | valid - 0.85119
Average accuracy on crossval is 0.89318
Std is 0.02457


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [14]:
model = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, model.predict(X_tr))
    score_val = accuracy_score(y_val, model.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 0.96902 | valid - 0.91716
train - 0.96572 | valid - 0.94675
train - 0.96309 | valid - 0.88757
train - 0.96572 | valid - 0.83432
train - 0.97100 | valid - 0.92308
train - 0.96770 | valid - 0.92308
train - 0.97299 | valid - 0.88690
train - 0.96772 | valid - 0.85714
train - 0.96904 | valid - 0.89881
train - 0.96706 | valid - 0.88690
Average accuracy on crossval is 0.89617
Std is 0.0317


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [15]:
model = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=21, min_samples_split=3, min_samples_leaf=1, max_leaf_nodes=100000, ccp_alpha=0)
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, model.predict(X_tr))
    score_val = accuracy_score(y_val, model.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 0.99670 | valid - 0.93491
train - 0.99341 | valid - 0.95266
train - 0.99670 | valid - 0.91716
train - 0.99341 | valid - 0.90533
train - 0.99275 | valid - 0.93491
train - 0.99341 | valid - 0.96450
train - 0.99736 | valid - 0.91667
train - 0.99407 | valid - 0.89881
train - 0.99275 | valid - 0.92262
train - 0.99407 | valid - 0.91667
Average accuracy on crossval is 0.92642
Std is 0.01944


In [16]:
model = RandomForestClassifier(n_estimators=250, max_depth=30, random_state=21)
kf = KFold(n_splits=10, shuffle=True, random_state=21)
scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model.fit(X_tr, y_tr)

    score_tr = accuracy_score(y_tr, model.predict(X_tr))
    score_val = accuracy_score(y_val, model.predict(X_val))
    
    scores.append(score_val)
    print(f"train - {score_tr:.05f} | valid - {score_val:.05f}")

scores = np.array(scores)

print(f"Average accuracy on crossval is {round(scores.mean(), 5)}")
print(f"Std is {round(scores.std(), 5)}")

train - 1.00000 | valid - 0.94675
train - 1.00000 | valid - 0.94675
train - 1.00000 | valid - 0.92308
train - 1.00000 | valid - 0.89941
train - 1.00000 | valid - 0.95858
train - 1.00000 | valid - 0.97041
train - 1.00000 | valid - 0.92857
train - 1.00000 | valid - 0.89881
train - 1.00000 | valid - 0.94048
train - 1.00000 | valid - 0.92857
Average accuracy on crossval is 0.93414
Std is 0.02216


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [17]:
model = RandomForestClassifier(n_estimators=250, max_depth=30, random_state=21)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy_score(y_test, predictions)

0.9349112426035503

In [18]:
test_analysis = pd.DataFrame({
    'y_true': y_test,
    'predictions': predictions
})

test_analysis['is_error'] = (test_analysis['y_true'] != test_analysis['predictions']).astype(int)

days_map = {
    0: '0. Понедельник',
    1: '1.     Вторник',
    2: '2.       Среда',
    3: '3.     Четверг',
    4: '4.     Пятница',
    5: '5.     Суббота',
    6: '6. Воскресенье'
}
test_analysis['day_name'] = test_analysis['y_true'].map(days_map)

error_by_day = test_analysis.groupby('day_name').agg(
    test_samples=('is_error', 'count'),  # Сколько тестовых объектов было в этот день
    total_errors=('is_error', 'sum'),     # Сколько раз модель ошиблаcь
    error_rate=('is_error', 'mean')       # Доля ошибок (Error Rate)
).reset_index()

error_by_day = error_by_day.sort_values(by='error_rate', ascending=False)

error_by_day['error_rate_%'] = (error_by_day['error_rate'] * 100).round(2)
error_by_day['accuracy_%'] = ((1 - error_by_day['error_rate']) * 100).round(2)

print(error_by_day[['day_name', 'test_samples', 'total_errors', 'error_rate_%', 'accuracy_%']].to_string(index=False))

      day_name  test_samples  total_errors  error_rate_%  accuracy_%
0. Понедельник            27             7         25.93       74.07
4.     Пятница            21             3         14.29       85.71
1.     Вторник            55             4          7.27       92.73
2.       Среда            30             2          6.67       93.33
5.     Суббота            54             3          5.56       94.44
3.     Четверг            80             2          2.50       97.50
6. Воскресенье            71             1          1.41       98.59


In [19]:
joblib.dump(model, '../data/RandomForestClassifier.joblib')

['../data/RandomForestClassifier.joblib']